# 🌿 Notebook Demostrativo: Parte 1 (EDA + Modelo Clásico) y Parte 2 (Agente LLM + RAG)
**SI3003 – Inteligencia Artificial | Universidad EAFIT (2026-1)**  
**Opción 3.1:** Agente de Calidad del Aire del Valle de Aburrá (SIATA / AMVA)

Este notebook ejecuta de principio a fin:
1. **Ingeniería de Datos y Plan B (`src/data_loader.py`):** Carga, limpieza e imputación de series históricas de $PM_{2.5}$ y meteorología del SIATA.
2. **Análisis Exploratorio de Datos (`src/eda.py`):** Estadísticas por estación y visualizaciones interactivas (Serie de tiempo con umbrales ICA, Matriz de correlación meteorológica y Dispersión Viento/Lluvia vs. $PM_{2.5}$).
3. **Entrenamiento, Evaluación vs. Baselines y Serialización del Modelo de la Parte 1 (`src/model_part1.py`):** Clasificador Gaussian Naive Bayes + Regresión Ridge + Filtro HMM exportado en `models/air_quality_model_part1.pkl`.
4. **Evaluación Experimental del Agente LLM + RAG (`src/evaluate_agent.py`):** Prueba de las 12 preguntas de benchmark verificando *tool calling*, citas del POECA y manejo de errores como observación.

In [ ]:
import sys
from pathlib import Path

# Asegurar que la raíz del repositorio esté en sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_air_quality_data
from src.eda import (
    compute_eda_summary,
    plot_pm25_timeseries_all_stations,
    plot_meteorological_correlation_matrix,
    plot_rain_wind_vs_pm25_scatter,
)
from src.model_part1 import ClassicalAirQualityModel
from src.agent_rag import AirQualityToolkit, AirQualityAgent
from src.evaluate_agent import run_agent_evaluation

## 1. Carga, Limpieza y Preprocesamiento de Datos (SIATA / Plan B)

In [ ]:
df = load_air_quality_data()
print(f"Dimensiones del dataset limpio: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head(10)

## 2. Análisis Exploratorio de Datos (EDA) y Visualizaciones Significativas

In [ ]:
summary_table = compute_eda_summary(df)
summary_table

In [ ]:
# Visualización 1: Serie de tiempo de PM2.5 por estaciones con bandas normativas (Resolución 2254 de 2017)
fig_ts = plot_pm25_timeseries_all_stations(df)
fig_ts.show()

In [ ]:
# Visualización 2A: Matriz de correlación entre PM2.5 y variables meteorológicas (Lluvia, Viento, Temperatura, Humedad)
fig_corr = plot_meteorological_correlation_matrix(df)
fig_corr.show()

In [ ]:
# Visualización 2B: Efecto de ventilación (viento) y lavado atmosférico (lluvia) sobre PM2.5
fig_scatter = plot_rain_wind_vs_pm25_scatter(df)
fig_scatter.show()

## 3. Entrenamiento, Evaluación vs. Líneas Base y Exportación del Modelo Clásico (Parte 1)

In [ ]:
model = ClassicalAirQualityModel()
metrics = model.fit_and_evaluate(df, train_ratio=0.75)
saved_path = model.save_model()

print(f"Modelo exportado exitosamente en: {saved_path}\n")
for k, v in metrics.items():
    print(f"{k}: {v}")

## 4. Demostración del Agente LLM + RAG y Suite de 12 Preguntas de Prueba (Parte 2)

In [ ]:
toolkit = AirQualityToolkit(
    data_csv_path=PROJECT_ROOT / "data" / "siata_pm25_historico.csv",
    docs_rag_dir=PROJECT_ROOT / "docs_rag",
)
agent = AirQualityAgent(toolkit=toolkit, t_max=5)

sample_resp = agent.run(
    "¿Cómo está el PM2.5 hoy en Itagüí, cuál es el pronóstico para mañana y qué recomienda el POECA para grupos sensibles?"
)
print("Iteraciones usadas:", sample_resp.iterations_used)
print("Herramientas llamadas:", [s.tool_name for s in sample_resp.steps])
print("\n--- RESPUESTA FINAL DEL AGENTE ---\n")
print(sample_resp.final_answer)

In [ ]:
# Evaluación sistemática sobre las 12 preguntas de prueba (Entregable 4)
df_eval = run_agent_evaluation(toolkit=toolkit, t_max=5)
df_eval